Imports which we will use in the deliverable:

In [11]:
from pathlib import Path
import re

Task 1: File ingestion and regular expressions.

Part 1 of Task 1: Robust file reading function.

In [5]:
def read_log_file(file_path: Path) -> tuple[str, str]:
    """Read a log file using either Latin-1 or UTF-8 encoding.
    
    Args:
        file_path: Path to the log file.
        
    Returns:
        A tuple containing decoded text and encoding used.
    """

    raw_data = file_path.read_bytes()

    try:
        return raw_data.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw_data.decode("latin-1"), "latin-1"

This function reades the bytes and first tries the utf-8 encoding.
If that fails, it will try the Latin-1 encoding.

Part 2 of Task 1: Log parser function.

In [13]:
def parse_log(file_text: str) -> list[dict[str, str | float]]:
    """Extract the transaction records from log text.
    
    Args:
        file_text: Log text including transactions and noise.
        
    Returns:
        A list of dictionaries with timestamp, register_id, item,
        price, currency and status. Prices as floats, other values
        as strings.
    """
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>POS_\d+)\]\s+"
        r"item=(?P<item>.+?)\s+"
        r"price=(?P<price>-?\d+(?:\.\d+)?)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions: list[dict[str, str | float]] = []

    for line in file_text.splitlines():
        match = pattern.fullmatch(line.strip())

        if match is not None:
            record: dict[str, str | float] = {
                "timestamp": match.group("timestamp"),
                "register_id": match.group("register_id"),
                "item": match.group("item"),
                "price": float(match.group("price")),
                "currency": match.group("currency"),
                "status": match.group("status"),
            }
            transactions.append(record)

    return transactions

This function extracts six field from each valid transaction line.
It will ignore lines which do not match the expected format. It will
return the transactions as a list of dictionaries.